# 📚🤖 [인공지능 자연어 처리 & 에이전트] 텍스트 분석, 코사인 유사도, 그리고 나만의 문학·규정집 에이전트 만들기

이 실습에서는 **LLM(거대 언어 모델)과 AI 에이전트의 원리**를 배우고, 
문학작품(소설 《동백꽃》, 《어린 왕자》)이나 학교생활 규정집을 바탕으로 **스스로 질문에 답하는 나만의 독서/규정 에이전트**를 직접 만들어봅니다.

> ### 🎯 이 모듈에서 배우는 것 (3줄 요약)
> 1. **문장을 숫자로 바꾸기**: 컴퓨터가 글을 작은 조각(토큰)으로 쪼개고 고유 번호표(ID)를 붙이는 원리
> 2. **문장을 화살표로 바꾸기**: 문장을 좌표(벡터)로 표현하고, 사잇각의 코사인(코사인 유사도)으로 관련성을 재는 수학
> 3. **나만의 AI 에이전트 만들기**: RAG(검색 증강 생성 — 책에서 근거를 먼저 **검색**한 뒤, 그 근거로 답을 **생성**하는 방식)로 원문 근거를 찾아 답하는 에이전트 완성
>
> ### 🎒 준비물 & 선수 지식
> - **준비물**: 인터넷이 되는 노트북, 구글 코랩(Colab) 접속
> - **선수 지식**: 파이썬 변수·리스트·함수 기초 (처음이어도 차근차근 따라오면 괜찮아요! 😊)
> - **선택**: Google AI Studio API 키 (없어도 마지막 선택 실습 전까지는 모두 체험할 수 있어요)

---

### 💡 오늘 우리가 배울 핵심 개념 3가지

1. **LLM (거대 언어 모델, Large Language Model)이란?**
   - 수억 권의 책과 인터넷 글을 학습하여, **"앞에 나온 단어들을 보고 다음에 올 가장 자연스러운 단어"**를 통계적으로 예측하는 인공지능의 거대한 언어 두뇌입니다.
2. **AI 에이전트 (AI Agent)란?**
   - 단순히 정해진 말만 하는 챗봇을 넘어, **목표(Goal)**를 부여받으면 주어진 문서나 도구를 **스스로 탐색(Search)**하고, **판단(Reason)**하여 문제를 해결하고 **행동(Act)**하는 자율적인 인공지능입니다.
3. **인공지능을 활용한 독서(AI Reading)의 혁신**:
   - 단순히 책을 눈으로만 읽는 '수동적 독서'에서 벗어나, 소설 속 등장인물(점순이, 어린 왕자, 여우)에게 **"너 그때 왜 그런 행동을 했어?"**라고 직접 인터뷰하고 질문하며 작가의 숨은 의도를 파헤치는 **'능동적 탐구 독서'**가 가능해집니다!

---

### 🗺️ 오늘 실습의 5단계 로드맵
1. **[기초 1단계] 문장 분석과 토큰화 (Tokenization)**: 컴퓨터는 글자를 어떻게 숫자로 쪼갤까?
2. **[기초 2단계] 문장의 수학적 표현 (벡터화, Vectorization)**: 글자를 공간 속 좌표로 바꾸기!
3. **[기초 3단계] 텍스트 검색의 핵심 수학 (코사인 유사도, Cosine Similarity)**: 두 문장의 각도를 재어 관련성 찾기!
4. **[기초 4단계] 지식 데이터베이스 구축**: 문학작품(《동백꽃》, 《어린 왕자》)과 학교생활규정집 탑재
5. **[본 실습] 나만의 검색 증강 생성(RAG) 에이전트 완성 및 실시간 Q&A 대화**


In [ ]:
#한글을 그래프에 보여주는 라이브러리
!pip install koreanize_matplotlib
import koreanize_matplotlib

## 0. 필요한 파이썬 도구 불러오기
텍스트 처리와 수학 연산(NumPy), 시각화(Matplotlib)에 필요한 표준 도구를 준비합니다.


In [ ]:
import numpy as np
import re
from collections import Counter
import math

# 시각화 도구 불러오기
try:
    import matplotlib.pyplot as plt
    has_plt = True
except ImportError:
    has_plt = False

print("✅ 파이썬 자연어 처리 & 에이전트 실습 도구가 준비되었습니다!")


---
## 1. [기초 1단계] 컴퓨터가 글을 읽는 첫걸음: 토큰화 (Tokenization)

### 💡 선생님의 개념 설명
- 컴퓨터는 사람처럼 '가', '나', '다' 같은 한글 글자의 모양을 직접 이해하지 못합니다.
- 그래서 컴퓨터는 문장을 의미가 있는 작은 조각들인 **토큰(Token, 단어 또는 형태소 조각)**으로 잘게 쪼갠 뒤, 각각의 토큰에 **고유한 번호표(ID 숫자)**를 붙여서 기억합니다.
- 이를 **토큰화(Tokenization)**라고 부르며, 모든 최신 LLM(GPT, Gemini, HyperCLOVA)이 글을 읽는 1단계 관문입니다.

> ✅ **체크포인트**: 문장을 토큰으로 쪼개고, 각 토큰에 고유한 번호표(ID)를 붙일 수 있다면 1단계 완료! 🎉


In [ ]:
# [실습 1-1] 문장을 단어 토큰으로 쪼개고 숫자 번호표(ID) 부여하기
# 💡 컴퓨터는 한글을 몰라요. 그래서 단어마다 '학번'처럼 고유 번호(ID)를 붙여 숫자로 기억합니다.

sample_text = "어제까지도 너는 내게 한마디도 건네지 않더니 오늘 점순이가 감자를 불쑥 내밀었다"

# 1. 띄어쓰기(공백)를 기준으로 문장을 단어 토큰으로 분리하기
tokens = sample_text.split()

# 2. 고유 단어 사전(Vocabulary) 만들기: 처음 보는 단어마다 숫자 번호(ID) 매기기
vocab = {}
next_id = 101  # 번호표는 101번부터 시작!

for word in tokens:
    if word not in vocab:  # 사전에 없는 새 단어라면
        vocab[word] = next_id
        next_id += 1

# 3. 문장을 컴퓨터가 이해하는 숫자(ID) 리스트로 변환하기
token_ids = [vocab[w] for w in tokens]

print(f"📖 원본 문장: '{sample_text}'\n")
print("🧩 1. 토큰 분리 결과:")
print(tokens)

print("\n🔢 2. 컴퓨터 단어 사전(단어 -> 숫자 ID):")
for word, wid in list(vocab.items())[:6]:
    print(f"   • '{word}' ➡️ ID: {wid}")

print("\n🤖 3. 컴퓨터가 실제로 기억하는 숫자 리스트:")
print(token_ids)


---
## 2. [기초 2단계] 문장의 수학적 표현: 단어 빈도 벡터 (Vectorization)

### 💡 선생님의 수학 설명
- 컴퓨터가 두 문장이 비슷한지 비교하려면, 문장을 수학의 **벡터(Vector, 좌표)**로 만들어야 합니다.
- 가장 직관적인 방법은 사전에 등록된 단어들이 **"해당 문장에 몇 번씩 등장했는가?"**를 세어서 리스트로 만드는 것입니다.
- 예를 들어 단어 사전이 `['감자', '점순이', '학교', '휴대폰']`일 때:
  - *"점순이가 감자를 주었다"* ➡️ `[1, 1, 0, 0]` (감자 1번, 점순이 1번)
  - *"학교에서 휴대폰을 보관했다"* ➡️ `[0, 0, 1, 1]` (학교 1번, 휴대폰 1번)
- 이렇게 문장을 숫자 좌표로 바꾸면, 수학적으로 거리를 재거나 각도를 계산할 수 있게 됩니다!

> ✅ **체크포인트**: "점순이가 감자를 주었다" 같은 문장을 단어 빈도 벡터 `[1, 1, 0, 0]`으로 바꿀 수 있다면 2단계 완료! 🎉


In [ ]:
# [실습 1-2] 3개의 서로 다른 문장을 좌표(벡터)로 변환해보기

sentences = [
    "점순이가 나에게 감자를 불쑥 내밀었다",       # 문장 1 (소설 동백꽃)
    "점순이가 감자를 건네며 호감을 표현했다",     # 문장 2 (문장 1과 매우 비슷한 의미)
    "학교 수업 중에는 스마트폰을 보관함에 넣는다" # 문장 3 (완전히 딴소리: 학교 규정)
]

# 전체 공통 단어 사전 만들기
all_words = sorted(list(set(" ".join(sentences).split())))
print(f"📚 전체 단어 사전 ({len(all_words)}개 단어):")
print(all_words)

# 각 문장을 단어 빈도 벡터로 변환하기 (Bag of Words: 단어들을 자루에 담아 개수를 세는 방식)
sentence_vectors = []
for s in sentences:
    words = s.split()
    # 단어 사전의 각 단어가 이 문장에 몇 번 나왔는지 세어 숫자 리스트로 만들기
    vec = [words.count(w) for w in all_words]
    sentence_vectors.append(np.array(vec))  # 넘파이 배열로 감싸서 계산하기 편하게!

print("\n📐 각 문장의 수학적 좌표 (벡터):")
for i, (s, v) in enumerate(zip(sentences, sentence_vectors)):
    print(f" • 문장 {i+1}: {v}  <-- '{s[:20]}...'")


---
## 3. [기초 3단계] 텍스트 검색의 핵심 수학: 코사인 유사도 (Cosine Similarity)

### 💡 선생님의 수학 설명: 왜 단순한 거리 대신 '각도'를 잴까요?
- 짧은 문장과 아주 긴 문장은 단순히 단어 개수(길이) 때문에 유클리드 거리가 멀어질 수 있습니다.
- 그래서 검색 엔진과 언어 모델에서는 두 화살표 사이의 **사잇각($\theta$)**을 측정하는 **코사인 유사도(Cosine Similarity)**를 사용합니다!
  $$\cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \|\vec{B}\|} = \frac{\sum A_i B_i}{\sqrt{\sum A_i^2} \sqrt{\sum B_i^2}}$$

- **코사인 유사도의 3가지 판정 기준**:
  - **$\theta = 0^\circ$ (사잇각 0도, 같은 방향)**: $\cos(0^\circ) = \mathbf{1.0}$ ➡️ **"완벽히 같은 주제! 정답 문장!"**
  - **$\theta = 45^\circ$ (사잇각 45도)**: $\cos(45^\circ) \approx \mathbf{0.7}$ ➡️ **"관련성이 꽤 높은 문장"**
  - **$\theta = 90^\circ$ (사잇각 90도, 직각)**: $\cos(90^\circ) = \mathbf{0.0}$ ➡️ **"전혀 엉뚱한 딴소리!"**

> ✅ **체크포인트**: "cos θ = 1.0이면 같은 주제, 0.0이면 딴소리!"라고 친구에게 설명할 수 있다면 3단계 완료! 🎉


In [ ]:
# [실습 1-3] 넘파이(NumPy)로 직접 코사인 유사도 계산하기 & 2D 각도 시각화
# 💡 코사인 유사도 = 두 화살표가 같은 방향을 볼수록 1에 가까워지는 '관련성 점수'

def cosine_similarity(v1, v2):
    # 두 벡터의 내적 (dot product): 같은 방향 성분이 얼마나 겹치는지 계산
    dot_product = np.dot(v1, v2)
    # 두 벡터의 크기(길이, norm)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

# 문장 1("점순이가 나에게 감자를 불쑥 내밀었다")을 기준으로 유사도 비교
v1 = sentence_vectors[0]
sim_1_2 = cosine_similarity(v1, sentence_vectors[1]) # 문장 2(비슷한 의미)
sim_1_3 = cosine_similarity(v1, sentence_vectors[2]) # 문장 3(학교 규정)

print("📊 [코사인 유사도 계산 결과]")
print(f" • 문장 1 vs 문장 2 (감자 호감 표현): cos θ = {sim_1_2:.4f} (일치도: {sim_1_2*100:.1f}%) ➡️ 관련성 매우 높음!")
print(f" • 문장 1 vs 문장 3 (스마트폰 규정): cos θ = {sim_1_3:.4f} (일치도: {sim_1_3*100:.1f}%) ➡️ 관련성 전혀 없음!")

# 2D 화살표 각도 시각화
if has_plt:
    fig, ax = plt.subplots(figsize=(7, 7))

    # 기준 원호
    angles = np.linspace(0, np.pi/2, 100)
    ax.plot(np.cos(angles), np.sin(angles), linestyle=':', color='gray')

    # 화살표 그리기
    # 질문(문장1 기준): 30도 위치
    ax.quiver(0, 0, np.cos(np.radians(30)), np.sin(np.radians(30)), angles='xy', scale_units='xy', scale=1, color='blue', label='질문: 점순이의 감자 (기준)')
    # 문장 2(유사함): 45도 위치 (각도 차이 15도 -> cos(15) = 0.96)
    ax.quiver(0, 0, np.cos(np.radians(45)), np.sin(np.radians(45)), angles='xy', scale_units='xy', scale=1, color='green', label=f'문장 2 (유사도: {sim_1_2:.2f})')
    # 문장 3(딴소리): 85도 위치 (각도 차이 55도 -> 직각에 가까움)
    ax.quiver(0, 0, np.cos(np.radians(85)), np.sin(np.radians(85)), angles='xy', scale_units='xy', scale=1, color='red', label=f'문장 3 (유사도: {sim_1_3:.2f})')

    ax.set_xlim(-0.1, 1.2)
    ax.set_ylim(-0.1, 1.2)
    ax.set_xlabel("단어 특징 차원 X", fontsize=11)
    ax.set_ylabel("단어 특징 차원 Y", fontsize=11)
    ax.set_title("📐 2D 공간 속 문장 벡터와 각도(θ) 비교", fontsize=13, fontweight='bold', pad=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')

    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ (구글 코랩 환경에서는 2D 각도기 화살표 그래프가 화면에 직접 시각화됩니다)")

print("👉 파란색 질문 화살표와 초록색 문장 2는 각도가 매우 좁아 같은 방향이지만,")
print("   빨간색 문장 3은 거의 90도 직각에 가까워 서로 완전히 다른 이야기임을 알 수 있습니다!")


---
## 4. [기초 4단계] 에이전트의 지식 베이스 구축 (문학작품 & 학교생활규정)

### 💡 왜 에이전트에게 '텍스트 데이터베이스'가 필요할까요?
- 최신 언어 모델도 우리 학교의 세부 규칙이나 특정 소설의 숨은 대목은 구체적으로 외우지 못하며, 잘못된 대답(환각, Hallucination)을 지어낼 수 있습니다.
- 그래서 에이전트에게 **정확한 원문 텍스트 데이터베이스**를 지식으로 쥐여주고, **"질문이 들어오면 반드시 이 텍스트 안에서 코사인 유사도로 증거를 찾아서 대답해!"**라고 지시하는 기법을 **검색 증강 생성(RAG, Retrieval-Augmented Generation)**이라고 부릅니다.
- 아래에 실습용으로 2가지 지식 데이터를 탑재합니다:
  1. **문학작품 지식**: 김유정의 단편소설 《동백꽃》
  2. **규정집 지식**: OO고등학교 《학생생활인권규정집》

> ✅ **체크포인트**: 에이전트의 지식 데이터베이스에 문학 작품과 규정집 문장이 잘 등록됐다면 4단계 완료! 🎉


In [ ]:
# [실습 1-4] 실습용 텍스트 데이터베이스 등록 (문학작품 vs 학교 규정집)
# 💡 코퍼스(corpus) = 에이전트가 읽고 기억할 '책'들의 모음. 데이터가 좋아야 답도 좋아져요!

# 1. 문학작품: 김유정 소설 《동백꽃》 핵심 문장 데이터베이스
dongbaek_corpus = [
    "오늘도 또 우리 수탉이 점순이네 큰 수탉에게 사납게 쪼이었다.",
    "산기슭에서 점순이가 닭을 몰고 오며 '느 집엔 이거 없지?' 하고 감자 세 개를 내밀었다.",
    "그것은 아주 김이 모락모락 나는 따끈따끈한 감자였지만, 나는 '난 감자 안 먹는다 너나 먹어라' 하고 밀쳐버렸다.",
    "그러자 점순이의 얼굴이 홍당무처럼 새빨개지더니 눈에 독이 올라서 홱 토라져 달아났다.",
    "그것은 점순이가 나에게 좋아하는 마음(호감)을 전하고 관심을 끌기 위해 집에서 몰래 쪄온 소중한 감자였던 것이다.",
    "거절당한 점순이는 자존심을 다쳐 우리 집 씨암탉과 수탉을 붙잡아 사납게 닭싸움을 붙이며 심술을 부리기 시작했다.",
    "어느 날 노란 동백꽃이 흐드러지게 핀 바위 틈에서 점순이와 나는 함께 넘어지며 서로의 마음을 확인하고 화해를 하였다.",
    "알싸하고 향긋한 노란 동백꽃의 냄새가 온 산골짜기에 가득 퍼져 나갔다."
]

# 2. 학교생활규정집 데이터베이스
school_rules_corpus = [
    "[제1조] 본 규정은 학생의 자율성과 책임감을 기르고 안전하고 존중받는 학교 공동체를 만드는 것을 목적으로 한다.",
    "[제5조] 등교 시간은 오전 8시 40분까지 지정된 교실에 입실해야 하며, 8시 41분 이후 등교는 지각으로 처리한다. 무단 지각 3회는 무단 결석 1회로 간주한다.",
    "[제8조] 수업 중 스마트폰 및 전자기기는 전원을 끄고 보관함에 넣어야 한다. 허가 없이 사용 적발 시 1차 주의, 2차 시 일주일간 담임교사 보관 조치한다.",
    "[제12조] 학생 복장 규정: 교복은 단정하게 착용하며, 동절기에는 교복 위에 방한용 외투나 패딩을 자유롭게 착용할 수 있다. 슬리퍼 등교는 금지한다.",
    "[제15조] 학생 상벌점제: 교내 봉사 및 선행 학생에게는 상점을 부여하며, 흡연·폭력·수업방해 학생에게는 벌점 및 학생선도위원회 회부를 적용한다.",
    "[제18조] 동아리 활동: 학생은 자율적으로 5인 이상의 회원을 모아 지도교사를 위촉하면 자율 동아리를 창설할 수 있다."
]

print(f"📖 1. 소설 《동백꽃》 데이터: 총 {len(dongbaek_corpus)}개 문장 등록 완료")
print(f"🏫 2. 《학생생활규정집》 데이터: 총 {len(school_rules_corpus)}개 조항 등록 완료")


---
## 5. [본 실습] 나만의 검색 증강 생성(RAG) 에이전트 만들기

### 🤖 에이전트의 3단계 문제 해결 루프
에이전트는 사용자의 질문을 받으면 다음과 같이 지능적으로 행동합니다:
1. **[1단계: 탐색 (Search)]**: 사용자의 질문과 문서 속 모든 문장 간의 **코사인 유사도**를 전수 조사하여, 가장 관련 깊은 **Top-1 근거 문장**을 찾아냅니다.
2. **[2단계: 판단 (Reason)]**: 찾아낸 근거 문장이 질문에 충분한 답이 되는지 확인합니다.
3. **[3단계: 답변 생성 및 출처 인용 (Act)]**: 정답과 함께 **"📖 참고한 원문 근거"**를 학생에게 투명하게 보여줍니다!

> ✅ **체크포인트**: 에이전트에게 질문을 던졌을 때 📖 원문 근거와 함께 답이 돌아오면 대성공! 🎉 원한다면 다음 선택 실습에서 진짜 LLM(Gemini) 두뇌를 연결해보세요.


In [ ]:
# [실습 1-5] 완전한 AI 텍스트 에이전트 함수 구현하기
# 💡 TextAgent = 질문을 받으면 책(코퍼스)에서 가장 비슷한 문장을 찾아 답하는 'AI 에이전트 설계도'

class TextAgent:
    def __init__(self, name, corpus):  # 에이전트 이름과 읽을 책(코퍼스)을 정하고 미리 준비하기
        self.name = name
        self.corpus = corpus
        # 코퍼스 전체 어휘 사전 구축
        all_words = []
        for doc in corpus:
            all_words.extend(re.sub(r'[^가-힣a-zA-Z0-9]', ' ', doc).split())
        self.vocab = sorted(list(set(all_words)))
        
        # 코퍼스 전체 문장 벡터화
        self.doc_vectors = [self._vectorize(doc) for doc in corpus]
        
    def _vectorize(self, text):
        # 문장을 단어 빈도 벡터로 바꾸는 내부 도구 (특수문자는 공백으로 정리)
        words = re.sub(r'[^가-힣a-zA-Z0-9]', ' ', text).split()
        return np.array([words.count(w) for w in self.vocab])
    
    def search(self, query, top_k=2):
        # [1단계: 탐색(Search)] 질문과 가장 비슷한 문장 top_k개를 코사인 유사도로 찾기
        q_vec = self._vectorize(query)
        scores = []
        for idx, d_vec in enumerate(self.doc_vectors):
            sim = cosine_similarity(q_vec, d_vec)
            scores.append((idx, sim, self.corpus[idx]))
            
        # 코사인 유사도 기준 내림차순 정렬
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]
    
    def answer(self, query):
        # [2단계: 판단(Reason)] + [3단계: 답변(Act)] 근거를 찾고 답과 출처를 함께 보여주기
        results = self.search(query, top_k=2)
        best_idx, best_score, best_chunk = results[0]
        
        print(f"\n{'='*60}")
        print(f"🤖 [{self.name}] 질문 답변 분석 중...")
        print(f"🙋 학생 질문: '{query}'")
        print(f"{'-'*60}")
        print(f"🔎 [1단계: 코사인 유사도 탐색 결과]")
        print(f"   • 1위 근거 문장 (유사도: {best_score*100:.1f}%): '{best_chunk}'")
        if len(results) > 1 and results[1][1] > 0:
            print(f"   • 2위 근거 문장 (유사도: {results[1][1]*100:.1f}%): '{results[1][2]}'")
            
        print(f"\n💬 [2단계: 에이전트의 최종 지능형 답변]")
        
        # 지능형 답변 템플릿
        if "동백꽃" in self.name:
            if "감자" in query:
                resp = "점순이가 감자를 건넨 진짜 이유는 주인공(나)에게 좋아하는 마음(호감)을 표현하고 관심을 끌기 위해서였습니다. 하지만 거절당하자 자존심을 다쳐 닭싸움을 붙이게 됩니다."
            elif "화해" in query or "바위" in query or "꽃" in query or "장소" in query:
                resp = "점순이와 나는 온 산골짜기에 향기가 가득 퍼진 노란 동백꽃이 흐드러지게 핀 바위 틈에서 함께 넘어지며 서로의 감정을 확인하고 화해하였습니다."
            elif "닭" in query or "싸움" in query:
                resp = "점순이는 감자를 거절당해 자존심이 상하자, 자신의 큰 수탉으로 하여금 우리 집 작은 수탉을 사납게 쪼아대며 괴롭히는 심술을 부렸습니다."
            else:
                resp = f"소설 원문에 따르면, '{best_chunk}' 라는 내용에서 질문하신 답을 찾을 수 있습니다."
        else:
            if "스마트폰" in query or "핸드폰" in query or "전자기기" in query or "휴대폰" in query:
                resp = "규정 제8조에 따라 수업 중 스마트폰 무단 사용 적발 시 1차는 주의 조치, 2차 적발 시에는 일주일간 담임교사 보관 조치가 이루어집니다."
            elif "지각" in query or "등교" in query:
                resp = "규정 제5조에 따라 등교 시간은 오전 8시 40분까지이며, 8시 41분 이후 입실은 지각 처리됩니다. 무단 지각 3회는 무단 결석 1회로 간주되니 주의해야 합니다."
            elif "패딩" in query or "외투" in query or "복장" in query or "교복" in query:
                resp = "규정 제12조에 따라 동절기에는 교복 위에 방한용 외투나 패딩을 자유롭게 착용할 수 있습니다. 단, 안전을 위해 슬리퍼 등교는 금지되어 있습니다."
            elif "동아리" in query:
                resp = "규정 제18조에 따라 5인 이상의 회원을 모으고 지도교사를 위촉하면 자율 동아리를 창설할 수 있습니다."
            else:
                resp = f"학생생활규정 원문에 따르면, '{best_chunk}' 조항에 따라 해당 규칙이 적용됩니다."
                
        print(f"   '{resp}'")
        print(f"\n📌 [출처 인용]: {best_chunk}")
        print(f"{'='*60}\n")

# 에이전트 2대 생성
dongbaek_agent = TextAgent("소설 《동백꽃》 문학 독서 에이전트", dongbaek_corpus)
rule_agent = TextAgent("OO고등학교 《학생생활규정》 에이전트", school_rules_corpus)

print("🎉 2대의 AI 에이전트(문학 에이전트 & 학교 규정 에이전트) 준비 완료!")


In [ ]:
# [실습 1-6] 에이전트에게 다양한 질문 던져보기
# 💡 이제 여러분이 인터뷰어! 궁금한 점을 마음껏 물어보고 에이전트의 '근거 찾기' 실력을 확인해보세요.

# 1. 문학작품 질문: 점순이의 심리 파악
dongbaek_agent.answer("점순이가 나에게 감자를 준 진짜 이유는 무엇인가요?")
dongbaek_agent.answer("두 사람이 화해한 장소는 어디인가요?")

# 2. 학교생활 규정집 질문: 지각 및 스마트폰 규정
rule_agent.answer("등교 시간은 몇 시까지이고 지각 기준은 어떻게 되나요?")
rule_agent.answer("수업 시간에 스마트폰을 쓰다가 걸리면 어떻게 되나요?")


---
## 6. [심화 탐구] 데이터(Data) vs AI(Brain)의 분리 & Google Gemini API 연동

### 💡 왜 '데이터'와 'AI 두뇌'를 명확히 구분해야 할까요?
* **📚 [1] 데이터 영역 (Data / Knowledge Base)**:
  - AI가 읽고 기억할 **'책(본문 텍스트)'**입니다.
  - 데이터가 없으면 아무리 똑똑한 인공지능이라도 소설 속 인물의 비밀을 알지 못해 엉뚱한 거짓말(환각)을 지어냅니다.
* **🧠 [2] AI 두뇌 영역 (AI / LLM Reasoning Engine)**:
  - 데이터를 읽고 학생의 질문 의도에 맞춰 생각하고 대답하는 **'독해자(LLM)'**입니다.
  - AI가 없으면 검색된 문장 조각만 덜렁 보여줄 뿐, 사람처럼 맥락을 짚어 대화하거나 캐릭터를 연기하지 못합니다.

> **🚨 교실 수업 보안 필수 원칙**:
> 선생님 또는 학생이 발급받은 Google API Key는 수업 중 실습할 때만 사용하세요.
> **수업이 끝나면 반드시 `del gemini_api_key` 및 세션 초기화로 키를 파기**해야 비용 청구나 오남용을 완벽하게 방지할 수 있습니다!


In [ ]:
# [실습 1-7] Google Gemini API로 나만의 LLM 에이전트 확장하기 (선택 실습)
import json
import urllib.request
import getpass

# 1. 교사용/실습용 Google Gemini API 키 입력 (입력 시 글자가 가려집니다)
#    getpass는 입력한 글자를 화면에 보이지 않게 가려주는 안전한 입력 도구예요.
# API 키가 없으면 엔터를 치고 넘어가면 오프라인 규칙 모드로 자동 동작합니다.
# 💡 선생님이 결제(빌링) 연결된 프로젝트의 키를 미리 아래에 넣어두면 학생은 입력 없이 바로 실습할 수 있어요.
#    (예: gemini_api_key = "AIzaSy..." — 단, 노트북 공유 시 키 노출에 주의하고 수업 후 키를 재발급하세요)
gemini_api_key = getpass.getpass("🔑 Google Gemini API Key 입력 (없으면 Enter): ").strip()

def ask_gemini_rag_agent(agent_name, persona_style, corpus, query, api_key=""):
    # 1단계: [데이터 영역] 코사인 유사도로 가장 관련 있는 원문 근거 2개 검색
    agent = TextAgent(agent_name, corpus)
    search_results = agent.search(query, top_k=2)
    best_chunk = search_results[0][2]
    best_score = search_results[0][1]
    
    print(f"\n{'='*60}")
    print(f"🤖 [{agent_name}] 질문 처리 중...")
    print(f"🙋 질문: '{query}'")
    print(f"📚 [데이터 영역] 코사인 유사도 1위 추출 근거 ({best_score*100:.1f}% 일치):")
    print(f"   \"{best_chunk}\"")
    print(f"{'-'*60}")
    
    # 2단계: [AI 두뇌 영역] Google Gemini API 호출 (다중 버전 및 모델 자동 호환)
    if api_key:
        prompt = f"""[역할]
너는 문학/규정 전문 AI 에이전트 '{agent_name}'이다.
말투 및 캐릭터 성격: {persona_style}

[엄격한 규칙]
1. 반드시 아래 [참고 본문 데이터]에 나온 사실만 근거로 답하라. (환각 금지)
2. 본문에 없는 내용은 '책에 나오지 않는 내용입니다'라고 답하라.
3. 캐릭터의 말투를 살려 생생하게 대답하라.

[참고 본문 데이터]
{best_chunk}

[학생 질문]
{query}
"""
        # 호환 가능한 모델 및 버전 후보 순차 시도 (무료 할당량이 넉넉한 flash 계열만 사용)
        # ⚠️ pro 계열 모델은 무료 할당량이 0이라 quota 에러가 발생하므로 제외했습니다.
        candidates = [
            ('v1beta', 'gemini-1.5-flash'),
            ('v1', 'gemini-1.5-flash'),
            ('v1beta', 'gemini-2.0-flash'),
            ('v1beta', 'gemini-2.5-flash'),
            ('v1beta', 'gemini-1.5-flash-latest'),
        ]
        
        answer = None
        success_model = ""
        data = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode('utf-8')
        
        for ver, mod in candidates:
            url = f"https://generativelanguage.googleapis.com/{ver}/models/{mod}:generateContent?key={api_key}"
            req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
            try:
                with urllib.request.urlopen(req, timeout=8) as response:
                    res_data = json.loads(response.read().decode('utf-8'))
                    if 'candidates' in res_data and res_data['candidates']:
                        answer = res_data['candidates'][0]['content']['parts'][0]['text']
                        success_model = f"{mod} ({ver})"
                        break
            except Exception:
                continue
                
        if answer:
            print(f"🧠 [AI 두뇌: Google Gemini {success_model} 추론 답변]:\n   {answer}")
        else:
            print("⚠️ API 호출 실패. 오프라인 규칙 모드로 대체합니다.")
            agent.answer(query)
    else:
        print("⚪ [오프라인 규칙 엔진 모드]")
        agent.answer(query)
        
    print(f"{'='*60}\n")

# 테스트 실행 (점순이 에이전트)
ask_gemini_rag_agent(
    agent_name="점순이",
    persona_style="새침데기 강원도 사투리로 톡톡 쏘아붙이지만 속마음은 다정하게",
    corpus=dongbaek_corpus,
    query="점순아 너 왜 나한테 감자 줬어?",
    api_key=gemini_api_key
)

# 🚨 [수업 종료 필수 코드] 실습 종료 후 메모리에서 API 키 파기하기!
if 'gemini_api_key' in locals():
    del gemini_api_key
    print("🔒 [수업 종료 완료] 메모리에 저장되었던 Gemini API Key가 안전하게 파기되었습니다.")


---
## 🔬 학생 탐구 실험 표 & 독서 퀴즈

### 1. 일반 챗봇 vs RAG 기반 AI 에이전트 비교표

| 항목 | 단순 대화 챗봇 (일반 챗봇) | 텍스트 검색 기반 AI 에이전트 (RAG) |
|---|---|---|
| **지식의 원천** | 사전에 미리 학습된 인터넷 데이터만 기억 | 우리가 제공한 소설, 시, 학급 규정집을 실시간 조회 |
| **답변의 신뢰도** | 모르는 내용은 그럴듯하게 거짓말함(환각) | **정확한 원문 문장(출처)을 100% 인용**하여 답변 |
| **수학적 원리** | 복잡한 확률 예측 | **토큰화 $\rightarrow$ 단어 벡터 $\rightarrow$ 코사인 유사도 검색** |
| **독서에서의 역할** | 줄거리 요약 정도만 가능 | 등장인물과의 인터뷰, 심리 분석, 논술 근거 찾기 가능 |

---

### 2. 오늘의 퀴즈 (복습)
1. **문장을 컴퓨터가 이해할 수 있는 작은 조각과 고유한 숫자 번호표(ID)로 쪼개는 과정을 무엇이라고 하나요?**
   - *정답: 토큰화 (Tokenization)*
2. **두 문장이 서로 같은 주제인지 딴소리인지 알아보기 위해 화살표 사이의 각도를 측정하는 수학 공식은 무엇인가요?**
   - *정답: 코사인 유사도 (Cosine Similarity)*
3. **코사인 유사도에서 사잇각이 $0^\circ$여서 $\cos(\theta) = 1.0$이 나왔다는 것은 무엇을 의미하나요?**
   - *정답: 두 문장의 방향이 완벽하게 일치한다 (가장 관련이 깊은 정답 문장이다)*
4. **거대 언어 모델이 모르는 사실을 지어내는 현상을 무엇이라고 부르며, 에이전트는 이를 어떻게 해결하나요?**
   - *정답: 환각(Hallucination) 현상 / 에이전트는 코사인 유사도로 원문 증거를 먼저 찾아낸 뒤 근거에 기반해서만 답변하여 해결한다.*
5. **내가 좋아하는 다른 소설책이나 우리 반 학급 규칙으로 에이전트를 만들려면 코퍼스(corpus)에 무엇을 넣으면 될까요?**
   - *정답: 내가 분석하고 싶은 소설의 문장들이나 학급 규칙 조항들을 리스트로 넣어주면 된다.*
